# K-Prototypes Clustering — Spotify Liked Songs

This notebook applies the **K-Prototypes** algorithm to cluster ~1,565 songs from a personal Spotify Liked Songs library into thematic groups. K-Prototypes is chosen over K-Means because the dataset contains a mix of **continuous** audio features (e.g., danceability, energy) and **categorical** features (e.g., musical key, mode, instrumentalness). K-Means relies on Euclidean distance, which is not meaningful for categorical data; K-Prototypes uses a hybrid dissimilarity measure that handles both types natively.

**Data source:** `spotifyLibrary_cleaned.csv` — preprocessed and MinMax-scaled output from the Spotify Data Cleaning notebook.

## 1. Setup & Library Imports

Import all required libraries. If `kmodes` is not installed in the current environment it will be installed automatically. `kmodes` provides the `KPrototypes` class, which extends the K-Means framework with support for categorical features using Huang's dissimilarity metric.

In [ ]:
import sys
import subprocess

try:
    from kmodes.kprototypes import KPrototypes
except ImportError:
    print("kmodes not found — installing...")
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'kmodes', '--quiet'])
    from kmodes.kprototypes import KPrototypes

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
np.random.seed(42)
plt.rcParams['figure.figsize'] = (12, 6)
sns.set_theme(style='whitegrid')

print("All libraries imported successfully.")
print(f"  pandas {pd.__version__}  |  numpy {np.__version__}  |  seaborn {sns.__version__}")

## 2. Load & Inspect the Cleaned Dataset

Load `spotifyLibrary_cleaned.csv`. This file was produced by the Data Cleaning notebook. It contains **11 audio features** per track — all numeric — with continuous features already MinMax-scaled to [0, 1]. No missing values are expected.

In [ ]:
df = pd.read_csv('../data_files/spotifyLibrary_cleaned.csv')

print(f"Dataset shape  : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Missing values : {df.isnull().sum().sum()}")
print(f"\nColumn dtypes:")
print(df.dtypes.to_string())
df.head()

## 3. Feature Identification — Continuous vs Categorical

K-Prototypes requires explicitly specifying which columns are **categorical** so it can apply the correct distance measure to each type:

| Type | Features | Notes |
|------|----------|-------|
| **Continuous** | danceability, energy, speechiness, acousticness, liveness, valence, tempo, duration_min | Float, MinMax-scaled [0,1] — Euclidean distance |
| **Categorical** | `key` (0–11), `mode` (0=minor, 1=major), `instrumentalness` (0=vocal, 1=instrumental) | Integer-encoded nominals — Hamming-style dissimilarity |

In [ ]:
CATEGORICAL_COLS = ['key', 'mode', 'instrumentalness']
CONTINUOUS_COLS  = [c for c in df.columns if c not in CATEGORICAL_COLS]
CAT_INDICES      = [list(df.columns).index(c) for c in CATEGORICAL_COLS]

print(f"Continuous features  ({len(CONTINUOUS_COLS)}): {CONTINUOUS_COLS}")
print(f"Categorical features ({len(CATEGORICAL_COLS)}): {CATEGORICAL_COLS}")
print(f"Categorical column indices : {CAT_INDICES}")
print()
for col in CATEGORICAL_COLS:
    print(f"  {col:20s} — unique values: {sorted(df[col].unique())}")

## 4. Prepare Data Matrix for K-Prototypes

K-Prototypes accepts a single NumPy array. Categorical columns are cast to **strings** so the library can distinguish them from continuous float columns during the dissimilarity computation. The original `df` is left unchanged for later visualisation.

In [ ]:
X = df.copy()
for col in CATEGORICAL_COLS:
    X[col] = X[col].astype(str)

X_matrix = X.to_numpy()

print(f"X_matrix shape : {X_matrix.shape}")
print(f"Array dtype    : {X_matrix.dtype}  (object — mixed types expected)")
print(f"\nSample row [0] : {X_matrix[0]}")

## 5. Elbow Method — Selecting the Optimal Number of Clusters

Run K-Prototypes for k = 2 through 12 and record the **total cost** (sum of intra-cluster dissimilarities). Plot cost vs k and look for the "elbow" — the inflection point where additional clusters yield diminishing reduction in cost.

- `init='Cao'` seeds centroids using Cao et al.'s method, which is more stable than random initialisation for mixed-type data.
- `n_init=1` is appropriate with Cao initialisation since it is deterministic.

In [ ]:
K_RANGE = range(2, 13)
costs   = []

print("Running elbow scan — fitting K-Prototypes for k = 2..12...")
print("(Each fit may take 10–30 seconds on ~1,500 records)\n")

for k in K_RANGE:
    kp = KPrototypes(n_clusters=k, init='Cao', n_init=1, verbose=0)
    kp.fit_predict(X_matrix, categorical=CAT_INDICES)
    costs.append(kp.cost_)
    print(f"  k={k:2d}  →  cost = {kp.cost_:>12,.2f}")

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(list(K_RANGE), costs,
        marker='o', linewidth=2.5, markersize=9,
        color='steelblue', markerfacecolor='white', markeredgewidth=2)
ax.set_xlabel('Number of Clusters (k)', fontsize=12)
ax.set_ylabel('K-Prototypes Cost (Sum of Dissimilarities)', fontsize=12)
ax.set_title('Elbow Method — Optimal k for K-Prototypes Clustering',
             fontsize=14, fontweight='bold')
ax.set_xticks(list(K_RANGE))
ax.grid(True, alpha=0.4)
sns.despine()
plt.tight_layout()
plt.savefig('../data_files/elbow_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nIdentify the elbow: the k where the cost curve transitions from steep to flat.")

## 6. Fit Final K-Prototypes Model

Based on the elbow plot, **k = 7** is selected — consistent with prior analysis showing musically meaningful separability at this granularity. The final model is fit with `verbose=1` to display iteration-level convergence progress.

In [ ]:
N_CLUSTERS = 7

print(f"Fitting K-Prototypes with k={N_CLUSTERS}, init='Cao'...\n")
kp_final = KPrototypes(n_clusters=N_CLUSTERS, init='Cao', n_init=1, verbose=1)
cluster_labels = kp_final.fit_predict(X_matrix, categorical=CAT_INDICES)

df['cluster'] = cluster_labels.astype(int)

print(f"\n{'='*45}")
print(f"  Final model cost : {kp_final.cost_:>14,.4f}")
print(f"  Songs clustered  : {len(cluster_labels):,}")
print(f"{'='*45}")
print(f"\nCluster sizes:")
print(df['cluster'].value_counts().sort_index().to_frame('count').to_string())

## 7. Cluster Size Distribution

Inspect how many songs fall into each cluster. Roughly balanced clusters suggest the algorithm found genuinely distinct groups. Heavily imbalanced clusters may indicate k should be increased, or that a cluster represents a rare but real song archetype (e.g., highly instrumental).

In [ ]:
palette = sns.color_palette('tab10', N_CLUSTERS)
cluster_counts = df['cluster'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(cluster_counts.index, cluster_counts.values,
              color=palette, edgecolor='white', linewidth=0.8)
ax.bar_label(bars, fmt='%d', padding=4, fontsize=11, fontweight='bold')
ax.set_xlabel('Cluster', fontsize=12)
ax.set_ylabel('Number of Songs', fontsize=12)
ax.set_title(f'Song Count per Cluster  (k = {N_CLUSTERS})',
             fontsize=14, fontweight='bold')
ax.set_xticks(range(N_CLUSTERS))
ax.set_ylim(0, cluster_counts.max() * 1.18)
sns.despine()
plt.tight_layout()
plt.show()

## 8. Cluster Feature Profile — Heatmap

Compute the **mean value of each continuous audio feature** within each cluster. The heatmap gives a high-level fingerprint of each cluster's musical character — for example, clusters with high energy + high danceability vs. high acousticness + low tempo.

In [ ]:
cluster_means = df.groupby('cluster')[CONTINUOUS_COLS].mean()

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(
    cluster_means.T,
    annot=True, fmt='.2f', cmap='YlOrRd',
    linewidths=0.5, linecolor='white',
    cbar_kws={'label': 'Mean Value (MinMax-scaled 0–1)'},
    ax=ax
)
ax.set_title('Mean Continuous Feature Values per Cluster',
             fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Cluster', fontsize=12)
ax.set_ylabel('Audio Feature', fontsize=12)
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('../data_files/cluster_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Continuous Feature Distributions — Violin Plots

Violin plots reveal the **full distribution shape** (not just the mean) of each continuous audio feature across all clusters. This makes it easy to spot tight vs. spread distributions and to identify the defining audio characteristics of each group.

In [ ]:
palette = sns.color_palette('tab10', N_CLUSTERS)

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, col in enumerate(CONTINUOUS_COLS):
    sns.violinplot(
        data=df, x='cluster', y=col,
        palette=palette, inner='box',
        linewidth=0.8, ax=axes[i]
    )
    axes[i].set_title(col.replace('_', ' ').title(),
                      fontsize=12, fontweight='bold')
    axes[i].set_xlabel('Cluster', fontsize=10)
    axes[i].set_ylabel('')

fig.suptitle('Audio Feature Distributions per Cluster',
             fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../data_files/violin_plots.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Categorical Feature Distributions per Cluster

Stacked bar charts show the **proportional breakdown** of each categorical feature within every cluster:

- **Key** — musical pitch class (C, C#, D … B)
- **Mode** — major (1) vs. minor (0)
- **Instrumentalness** — vocal (0) vs. instrumental (1)

These distributions highlight whether clusters gravitate toward particular tonalities, scales, or production styles.

In [ ]:
KEY_LABELS = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']

fig, axes = plt.subplots(1, 3, figsize=(21, 6))

# ── Musical key ──────────────────────────────────────────────────────────────
key_pct = (df.groupby(['cluster', 'key']).size()
             .unstack(fill_value=0)
             .reindex(columns=range(12), fill_value=0)
             .apply(lambda r: r / r.sum(), axis=1))
key_pct.columns = KEY_LABELS
key_pct.plot(kind='bar', stacked=True, ax=axes[0], colormap='tab20')
axes[0].set_title('Musical Key Distribution', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Cluster')
axes[0].set_ylabel('Proportion')
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(title='Key', bbox_to_anchor=(1.0, 1.0), fontsize=8)

# ── Mode (major / minor) ─────────────────────────────────────────────────────
mode_pct = (df.groupby(['cluster', 'mode']).size()
              .unstack(fill_value=0))
for c in [0, 1]:
    if c not in mode_pct.columns:
        mode_pct[c] = 0
mode_pct = mode_pct[[0, 1]].apply(lambda r: r / r.sum(), axis=1)
mode_pct.columns = ['Minor (0)', 'Major (1)']
mode_pct.plot(kind='bar', stacked=True, ax=axes[1], color=['#5588CC', '#EE7733'])
axes[1].set_title('Mode Distribution (Major / Minor)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Cluster')
axes[1].set_ylabel('Proportion')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(title='Mode')

# ── Instrumentalness (vocal / instrumental) ───────────────────────────────────
instr_pct = (df.groupby(['cluster', 'instrumentalness']).size()
               .unstack(fill_value=0))
for c in [0, 1]:
    if c not in instr_pct.columns:
        instr_pct[c] = 0
instr_pct = instr_pct[[0, 1]].apply(lambda r: r / r.sum(), axis=1)
instr_pct.columns = ['Vocal (0)', 'Instrumental (1)']
instr_pct.plot(kind='bar', stacked=True, ax=axes[2], color=['#44AA99', '#AA3377'])
axes[2].set_title('Instrumentalness per Cluster', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Cluster')
axes[2].set_ylabel('Proportion')
axes[2].tick_params(axis='x', rotation=0)
axes[2].legend(title='Type')

plt.tight_layout()
plt.savefig('../data_files/categorical_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Export Clustered Dataset

Save the final DataFrame — original features plus the `cluster` label column — to `songDataKProto.csv`. This file can be used downstream for Spotify playlist creation, further qualitative analysis, or word-cloud generation per cluster.

In [ ]:
output_path = '../data_files/songDataKProto.csv'
df.to_csv(output_path, index=False)

print(f"Saved to      : {output_path}")
print(f"Shape         : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Columns       : {list(df.columns)}")
print()
df.head(10)